* DSC 550-T301 Data Mining
* Week 3 Exercise
* Peter Lozano

# Import packages


In [7]:
import pandas as pd
from textblob import TextBlob
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sklearn.metrics import accuracy_score

# Part 1

## 1. Import the movie review data as a data frame and ensure that the data is loaded properly.

In [8]:
# 1. Load data (replace 'reviews.csv' with your actual file name)
df = pd.read_csv('Data/labeledTrainData.tsv', sep='\t')
df.head()

,id,sentiment,review
0,5814_8,1,With all this stuff going down at the moment w...
1,2381_9,1,"\The Classic War of the Worlds\"" by Timothy Hi..."
2,7759_3,0,The film starts with a manager (Nicholas Bell)...
3,3630_4,0,It must be assumed that those who praised this...
4,9495_8,1,Superbly trashy and wondrously unpretentious 8...


The data frame has been loaded successfully. It contains 3 columns: 'id', 'sentiment', and 'review'.

However, the sentiment column only contains binary values indicating positive (1) or negative (0) reviews which is detailed in the source data documentation: [Bag of Words Meet Bags of Popcorn](https://www.kaggle.com/c/word2vec-nlp-tutorial/data)

## 2. How many of each positive and negative reviews are there?

In [11]:
# Count the number of positive and negative reviews
df['sentiment'].map({0: 'negative', 1: 'positive'}).value_counts()

sentiment
positive    12500
negative    12500
Name: count, dtype: int64

I don't want to convert the sentiment column to human-readable labels permanently; I just want to count them but still make the results readable.

What is interesting is that the counts for both positive and negative reviews are exactly the same. This indicates that the dataset is perfectly balanced in terms of sentiment distribution. And possibly just a controlled sample of a larger dataset.

## 3. Use TextBlob to classify each movie review as positive or negative. Assume that a polarity score greater than or equal to zero is a positive sentiment and less than 0 is a negative sentiment.

In [13]:
# TextBlob Classification
df['textblob_score'] = df['review'].apply(
    lambda x:
        # Calculate the polarity score of the review
        TextBlob(str(x)).sentiment.polarity
)

# Convert the polarity score to a binary prediction (1 for positive, 0 for negative)
df['textblob_pred'] = df['textblob_score'].apply(lambda x: 1 if x >= 0 else 0)

# Count the number of positive and negative predictions made by TextBlob
df['textblob_pred'].map({0: 'negative', 1: 'positive'}).value_counts()

textblob_pred
positive    19017
negative     5983
Name: count, dtype: int64

The results are pretty off compared to the actual sentiment labels. It appears to have a bias towards predicting positive sentiment.

## 4. Check the accuracy of this model. Is this model better than random guessing?

In [14]:
# 4. Check TextBlob Accuracy
tb_acc = accuracy_score(df['sentiment'], df['textblob_pred'])
print(f"TextBlob Accuracy: {tb_acc:.2f}")

TextBlob Accuracy: 0.69


Yes! The TextBlob model is better than random guessing, but it is not perfect and has a tendency to predict positive sentiment more often as we saw in the actual results above.

## 5. For up to five points extra credit, use another prebuilt text sentiment analyzer, e.g., VADER, and repeat steps (3) and (4).

Using VADER sentiment analysis is a bit more complex than TextBlob because it considers the intensity of sentiment and provides a compound score that needs to be interpreted to classify sentiment as positive or negative. For example, VADER evaluates intensity modifiers such as "very" or "extremely" to adjust the sentiment score accordingly.

The result should be a bit more nuanced and potentially more accurate than TextBlob, especially for sentences with mixed sentiment or intensity modifiers.

In [16]:
# VADER Classification (Updated to output 1 and 0)
vader = SentimentIntensityAnalyzer()
df['vader_score'] = df['review'].apply(lambda x: vader.polarity_scores(str(x))['compound'])
df['vader_pred'] = df['vader_score'].apply(lambda x: 1 if x >= 0 else 0)
# Count the number of positive and negative predictions made by VADER
df['vader_pred'].map({0: 'negative', 1: 'positive'}).value_counts()

vader_pred
positive    16611
negative     8389
Name: count, dtype: int64

Based on the counts, it appears that VADER is slightly less positive in its predictions compared to TextBlob. However, it could be false positives or false negatives that I won't be able to see unless I apply an accuracy check.

In [17]:
vader_acc = accuracy_score(df['sentiment'], df['vader_pred'])
print(f"VADER Accuracy: {vader_acc:.2f}")

VADER Accuracy: 0.69


Surprisingly, it fairs no better or worse than TextBlob in terms of accuracy, indicating that both sentiment analyzers have their strengths and weaknesses depending on the context of the reviews.

I could improve the VADER lexicon by adding domain-specific sentiment words that are relevant to the context of the reviews. But, I'm not familiar enough with the domain to do that effectively.

# Part 2 Prepping Text for a Custom Model

## Import packages

In [ ]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

## 1. Convert all text to lowercase letters.

*See step 4 for a combined function*

## 2. Remove punctuation and special characters from the text

*See step 4 for a combined function*

## 3. Remove stop words.

*See step 4 for a combined function*

## 4. Apply Natural Language Toolkit (NLTK) PorterStemmer.

Creating a function to clean text data that will handle both lowercase conversion, punctuation removal and stop word removal, as well as applying stemming using the PorterStemmer.

To do this, I need to first download the stopwords from the NLTK library.

In [19]:
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\peter\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Variable `stop_words` contains the set of English stop words used to filter out common words that do not carry significant meaning in text analysis.

In [20]:
# Function to clean text by lowercasing
# Removing punctuation, special characters
# Stop words removal and stemming
def clean_text(text):
    # 1 & 2. Lowercase and remove punctuation/special characters
    text = re.sub(
        # Removing punctuation and special characters
        r'[^\w\s]', '',
        # Using lower() function to convert text
        str(text).lower()
    )
    # 3 & 4. Remove stop words and apply PorterStemmer
    words = [
        # Apply stemming to non-stop words
        PorterStemmer().stem(w)
        # Split the passed text into words
        for w in text.split()
        # Filter out the stop words
        if w not in stop_words
    ]
    # Joining the cleaned words back into a single string
    return ' '.join(words)

# Apply the function to 'review' column, creating a new column
df['cleaned_review'] = df['review'].apply(clean_text)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\peter\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


I'm creating a new column to avoid altering the original review column. This way I can continue to alter the model input without losing the original data.

I start by removing punctuation and special characters to avoid any noise in the text prior to further processing such as stop words removal and stemming. I then convert the text to lowercase to maintain consistency. Finally, I remove stop words first then apply the Porter stemmer to reduce words to their root forms. Although the function appears to have Porter stemming applied first, it is only until after it check if the stop words are present that stemming is actually applied.

## 5. Create a bag-of-words matrix from your stemmed text (output from (4)), where each row is a word-count vector for a single movie review (see sections 5.3 & 6.8 in the Machine Learning with Python Cookbook). Display the dimensions of your bag-of-words matrix. The number of rows in this matrix should be the same as the number of rows in your original data frame.

Using CountVectorizer (6.9 in the Machine Learning with Python Cookbook) to create a bag-of-words matrix from the cleaned reviews.

CountVectorizer is used to convert the collection of reviews into a matrix of token counts, where each row represents a review and each column represents a unique word from the entire corpus.

The fit_transform method of CountVectorizer is used to learn the vocabulary from the cleaned reviews and simultaneously transform the reviews into the bag-of-words matrix.

In [24]:
# 5. Bag-of-Words Matrix
count_vec = CountVectorizer()
bow_matrix = count_vec.fit_transform(df['cleaned_review'])
print(f"Bag-of-Words dimensions: {bow_matrix.shape[0]:,} rows x {bow_matrix.shape[1]:,} columns")

Bag-of-Words dimensions: 25,000 rows x 92,532 columns


The result of **25,000** is the number of rows in the bag-of-words matrix, which corresponds to the number of movie reviews in the original data frame. This corresponds perfectly to the **12,500** positive and negative reviews in the dataset.

**92,532** is the number of unique words (features) in the bag-of-words matrix, which corresponds to the size of the vocabulary learned from the cleaned reviews.

This is a very wide data matrix!

## 6. Create a term frequency-inverse document frequency (tf-idf) matrix from your stemmed text, for your movie reviews (see section 6.9 in the Machine Learning with Python Cookbook). Display the dimensions of your tf-idf matrix. These dimensions should be the same as your bag-of-words matrix.

Section 6.10 covers the `TfidfVectorizer` and how to create a term frequency-inverse document frequency (tf-idf) matrix from text data. It compares the frequency of the word in a document (tweet, movie review, speech transcript) with the frequency of the word in all other documents using the inverse document frequency (IDF) metric. This helps to downweight common words that appear in many documents and highlight words that are more unique to each document.

In [25]:
# TF-IDF Matrix
tfidf_vec = TfidfVectorizer()
tfidf_matrix = tfidf_vec.fit_transform(df['cleaned_review'])
print(f"TF-IDF dimensions: {tfidf_matrix.shape[0]:,} rows x {tfidf_matrix.shape[1]:,} columns")

TF-IDF dimensions: 25,000 rows x 92,532 columns


The set up is very similar to how I setup the bag-of-words matrix using the `CountVectorizer`, except that here I am using the `TfidfVectorizer` to create a tf-idf matrix.

I can also check the vocabulary learned by the `TfidfVectorizer` using the `vocabulary_` attribute. This tells me which terms were extracted from the text and their corresponding feature indices in the tf-idf matrix.



In [ ]:
# Limit results to the first 10 items
list(tfidf_vec.vocabulary_.items())[:10]

[('stuff', 78431),
 ('go', 34036),
 ('moment', 53274),
 ('mj', 53050),
 ('ive', 42552),
 ('start', 77281),
 ('listen', 47712),
 ('music', 54873),
 ('watch', 88643),
 ('odd', 57954)]

I will revisit this preporcessed text data to build a custom model in the future. So far, this is just the preprocessing step for a linear model.